# BeetleLM quickstart

Train a tiny transformer language model end-to-end on CPU with **zero downloads**,
using the `beetlelm.toy` path. This walks through: install → train → inspect the
checkpoint → change one variable → re-run → diff against the shipped baseline.

`beetlelm.toy` reuses the *real* BeetleLM PicoDecoder architecture and byte
tokenizer; it just skips the cluster / sweep / continual-learning machinery so it
runs on a laptop. The full research trainer is `beetlelm-train` (see the README).

Run this notebook from the repository root so `configs/` and `data/` resolve.


## 1. Install

Light install (just the toy path):

```bash
pip install 'torch>=2.5.1' transformers numpy pyyaml click
pip install -e . --no-deps
```

or the full stack with `pip install -e .`. The cell below assumes `beetlelm` imports.


In [ ]:
import beetlelm
print('beetlelm version:', beetlelm.__version__)

## 2. Train the headline toy model

`configs/tiny_cpu.yaml` trains a ~0.72M-parameter PicoDecoder on the in-repo
bilingual byte corpus for 200 steps. On a laptop CPU this takes ~15 seconds.


In [ ]:
from beetlelm.toy import train
results = train('configs/tiny_cpu.yaml')
print('final train loss:', results['final_loss'])
print('final val loss:  ', results['final_val_loss'])
print('parameters:      ', results['n_params'])

Random loss over the 256-way byte vocabulary is `ln(256) ≈ 5.55`.
A successful run drops well below that, showing the model is learning.


## 3. Inspect the checkpoint

Checkpoints are written in HuggingFace format, so they reload with `from_pretrained`.


In [ ]:
import json, pathlib
ckpt = pathlib.Path(results['checkpoint_dir'])
print('files:', sorted(p.name for p in ckpt.iterdir()))
print('metrics final_loss:', json.loads((ckpt/'metrics.json').read_text())['final_loss'])

In [ ]:
from beetlelm.pico_decoder.pico_decoder import PicoDecoderHF
from beetlelm.bytes import ByteTokenizerManager
model = PicoDecoderHF.from_pretrained(results['checkpoint_dir'])
tok = ByteTokenizerManager.from_pretrained(results['checkpoint_dir'])
print('reloaded model with vocab_size =', model.config.vocab_size)

### Sample a few bytes from the trained model

A tiny greedy continuation — the toy model is small, so this is just a sanity check.


In [ ]:
import torch
model.eval()
ids = torch.tensor([tok.encode('El escarabajo')], dtype=torch.long)
with torch.no_grad():
    for _ in range(40):
        logits = model(input_ids=ids).logits
        nxt = int(logits[0, -1].argmax())
        ids = torch.cat([ids, torch.tensor([[nxt]])], dim=1)
print(repr(tok.decode(ids[0].tolist())))

## 4. Change one variable and re-run

Everything is controlled by the config or by `overrides`. Here we change the
seed and the model width and compare the final loss.


In [ ]:
baseline = train('configs/tiny_cpu.yaml', overrides={'training': {'output_dir': 'runs/seed42'}})
variant  = train('configs/tiny_cpu.yaml', overrides={
    'training': {'seed': 7, 'output_dir': 'runs/seed7'},
    'model':    {'d_model': 192},
})
print('baseline (seed 42, d_model 128):', baseline['final_loss'])
print('variant  (seed  7, d_model 192):', variant['final_loss'])

## 5. Reproduce vs. the shipped baseline

`baselines/tiny_cpu_baseline.json` records the expected metrics. Diff your run against it.


In [ ]:
expected = json.load(open('baselines/tiny_cpu_baseline.json'))['runs']['tiny_cpu']
got = train('configs/tiny_cpu.yaml')['final_loss']
delta = abs(got - expected['final_loss'])
print(f"expected={expected['final_loss']}  got={got:.6f}  |delta|={delta:.4f}")
print('within tolerance' if delta <= expected['tolerance'] else 'OUTSIDE tolerance')

## Scaling up to the full framework

- Swap in your own corpus: point `data.train_file` at any UTF-8 text file.
- Scale the toy model: raise `n_layers`, `d_model`, `max_steps`; set `training.device: cuda`.
- For the real bilingual/continual-learning pipeline (Lightning, HF datasets,
  the architecture zoo), use the `beetlelm-train` CLI with the configs under `configs/`.
- The checkpoint is standard HuggingFace — push it to the Hub or load it anywhere.
